# Check Forcing File Calendar & Leap Days
Reads `forcing_dir` from `config_aral.yaml`, then uses `ncdump` to inspect calendar metadata and detect Feb 29 dates directly from disk.

In [29]:
import subprocess
import re
import datetime
from pathlib import Path
import yaml


CONFIG_YAML = Path("../../config_aral.yaml")
# ----------------

with open(CONFIG_YAML) as f:
    config = yaml.safe_load(f)

project_root  = Path(config["paths"]["project_root"])
forcing_base  = (project_root / config["paths"]["forcing_dir"]).resolve()
output_txt    = forcing_base / "calendar_check.txt"

forcing_files = sorted(forcing_base.rglob("work/diagnostic/script/*.nc"))
print(f"Forcing base : {forcing_base}")
print(f"Found {len(forcing_files)} NetCDF files")

Forcing base : /home/avandervee3/aral_sea_full_project/data/forcing
Found 190 NetCDF files


In [30]:
def ncdump_header(filepath):
    result = subprocess.run(["ncdump", "-h", str(filepath)], capture_output=True, text=True)
    return result.stdout

def ncdump_time(filepath):
    result = subprocess.run(["ncdump", "-v", "time", str(filepath)], capture_output=True, text=True)
    return result.stdout

def extract_attr(header, attr):
    match = re.search(rf'{attr}\s*=\s*"([^"]+)"', header)
    return match.group(1) if match else "NOT FOUND"

def find_feb29(time_dump, units):
    match = re.search(r'days since (\d{4}-\d{2}-\d{2})', units)
    if not match:
        return []
    origin = datetime.date.fromisoformat(match.group(1))
    data_section = time_dump.split("data:")[-1]
    numbers = re.findall(r'[\d]+(?:\.\d+)?', data_section)
    return [
        str(origin + datetime.timedelta(days=int(float(n))))
        for n in numbers
        if (origin + datetime.timedelta(days=int(float(n)))).month == 2
        and (origin + datetime.timedelta(days=int(float(n)))).day == 29
    ]

In [31]:
lines = []
lines.append("=" * 70)
lines.append("FORCING FILE CALENDAR CHECK")
lines.append(f"Directory : {forcing_base}")
lines.append(f"Files     : {len(forcing_files)}")
lines.append("=" * 70)

for f in forcing_files:
    # lines.append(f"\n--- {f.name} ---")
    lines.append(f"\n--- {f.relative_to(forcing_base)} ---")
    header   = ncdump_header(f)
    calendar = extract_attr(header, "calendar")
    units    = extract_attr(header, "time:units")
    lines.append(f"  calendar : {calendar}")
    lines.append(f"  units    : {units}")

    feb29 = find_feb29(ncdump_time(f), units)
    if feb29:
        lines.append(f"  WARNING  : Feb 29 dates found ({len(feb29)}): {feb29}")
    else:
        lines.append(f"  OK       : No Feb 29 dates in time axis")

lines.append("\n" + "=" * 70)
output_txt.write_text("\n".join(lines))
print(f"Written to {output_txt}")
print("\n".join(lines))

Written to /home/avandervee3/aral_sea_full_project/data/forcing/calendar_check.txt
FORCING FILE CALENDAR CHECK
Directory : /home/avandervee3/aral_sea_full_project/data/forcing
Files     : 190

--- CMIP6/future/bias_corrected/CanESM5/ssp126/r1i1p1f1/2015-2100/AralSea_basin/work/diagnostic/script/pcrglobwb_CMIP6_CanESM5_day_ssp126_r1i1p1f1_pr_gn_2015-2100_AralSea_basin.nc ---
  calendar : 365_day
  units    : days since 1850-01-01

--- CMIP6/future/bias_corrected/CanESM5/ssp126/r1i1p1f1/2015-2100/AralSea_basin/work/diagnostic/script/pcrglobwb_CMIP6_CanESM5_day_ssp126_r1i1p1f1_tas_gn_2015-2100_AralSea_basin.nc ---
  calendar : 365_day
  units    : days since 1850-01-01

--- CMIP6/future/bias_corrected/CanESM5/ssp245/r1i1p1f1/2015-2100/AralSea_basin/work/diagnostic/script/pcrglobwb_CMIP6_CanESM5_day_ssp245_r1i1p1f1_pr_gn_2015-2100_AralSea_basin.nc ---
  calendar : 365_day
  units    : days since 1850-01-01

--- CMIP6/future/bias_corrected/CanESM5/ssp245/r1i1p1f1/2015-2100/AralSea_basin/wor

In [32]:
import subprocess, re, datetime

filepath = forcing_base / "ERA5/raw/1940-2020/AralSea_basin/work/diagnostic/script/pcrglobwb_OBS6_ERA5_reanaly_1_day_pr_1940-2020_AralSea_basin.nc"
filepath_can = "/home/avandervee3/aral_sea_full_project/data/forcing/CMIP6/historical/regridded/CanESM5/r1i1p1f1/1940-2014/AralSea_basin/work/diagnostic/script/pcrglobwb_CMIP6_CanESM5_day_historical_r1i1p1f1_pr_gn_1940-2014_AralSea_basin.nc"
filepath_miroc6 = "/home/avandervee3/aral_sea_full_project/data/forcing/CMIP6/historical/regridded/MIROC6/r1i1p1f1/1940-2014/AralSea_basin/work/diagnostic/script/pcrglobwb_CMIP6_MIROC6_day_historical_r1i1p1f1_pr_gn_1940-2014_AralSea_basin.nc"
result = subprocess.run(["ncdump", "-v", "time", filepath_miroc6], capture_output=True, text=True)
time_dump = result.stdout

units_match = re.search(r'time:units = "([^"]+)"', time_dump)
units = units_match.group(1)

date_str = units.split("days since ")[-1].strip().split(".")[0]
try:
    origin = datetime.datetime.strptime(date_str, "%Y-%m-%d %H:%M:%S").date()
except ValueError:
    origin = datetime.datetime.strptime(date_str, "%Y-%m-%d").date()

data_section = time_dump.split("data:")[-1]
numbers = re.findall(r'[\d]+(?:\.\d+)?', data_section)

feb29s = [
    str(origin + datetime.timedelta(days=int(float(n))))
    for n in numbers
    if (d := origin + datetime.timedelta(days=int(float(n)))).month == 2 and d.day == 29
]

print(f"Units: {units}")
print(f"Feb 29 count: {len(feb29s)}")
print(feb29s)

Units: days since 1850-01-01
Feb 29 count: 0
[]


In [33]:
import cftime

calendar = "365_day"
data_section = time_dump.split("data:")[-1]
numbers = re.findall(r'[\d]+(?:\.\d+)?', data_section)
offsets = [int(float(n)) for n in numbers]

dates = cftime.num2date(offsets, units=units, calendar=calendar)
feb29s = [str(d) for d in dates if d.month == 2 and d.day == 29]

print(f"Feb 29 count: {len(feb29s)}")
print(feb29s)

Feb 29 count: 0
[]


In [34]:
print(result.returncode)
print(result.stderr)

0



In [35]:
print(units)

days since 1850-01-01


In [36]:
forcing_base

PosixPath('/home/avandervee3/aral_sea_full_project/data/forcing')

In [37]:
filepath = forcing_base / "CMIP6/historical/raw/MIROC6/r1i1p1f1/1940-2014/AralSea_basin/work/diagnostic/script/pcrglobwb_CMIP6_MIROC6_day_historical_r1i1p1f1_pr_gn_1940-2014_AralSea_basin.nc"

result = subprocess.run(["ncdump", "-v", "time", str(filepath)], capture_output=True, text=True)
time_dump = result.stdout

units_match = re.search(r'time:units = "([^"]+)"', time_dump)
units = units_match.group(1)

import cftime
data_section = time_dump.split("data:")[-1]
offsets = [int(float(n)) for n in re.findall(r'[\d]+(?:\.\d+)?', data_section)]
dates = cftime.num2date(offsets, units=units, calendar="standard")
feb29s = [str(d) for d in dates if d.month == 2 and d.day == 29]

print(f"Units: {units}")
print(f"Total timesteps: {len(offsets)}")
print(f"Feb 29 count: {len(feb29s)}")
print(feb29s)

Units: days since 1850-1-1 00:00:00
Total timesteps: 27759
Feb 29 count: 19
['1940-02-29 00:00:00', '1944-02-29 00:00:00', '1948-02-29 00:00:00', '1952-02-29 00:00:00', '1956-02-29 00:00:00', '1960-02-29 00:00:00', '1964-02-29 00:00:00', '1968-02-29 00:00:00', '1972-02-29 00:00:00', '1976-02-29 00:00:00', '1980-02-29 00:00:00', '1984-02-29 00:00:00', '1988-02-29 00:00:00', '1992-02-29 00:00:00', '1996-02-29 00:00:00', '2000-02-29 00:00:00', '2004-02-29 00:00:00', '2008-02-29 00:00:00', '2012-02-29 00:00:00']


In [38]:
filepath = forcing_base / "CMIP6/historical/regridded/CanESM5/r1i1p1f1/1940-2014/AralSea_basin/work/diagnostic/script/pcrglobwb_CMIP6_CanESM5_day_historical_r1i1p1f1_pr_gn_1940-2014_AralSea_basin.nc"

result = subprocess.run(["ncdump", "-v", "time", str(filepath)], capture_output=True, text=True)
time_dump = result.stdout
units_match = re.search(r'time:units = "([^"]+)"', time_dump)
units = units_match.group(1)

offsets = [int(float(n)) for n in re.findall(r'[\d]+(?:\.\d+)?', time_dump.split("data:")[-1])]
dates = cftime.num2date(offsets, units=units, calendar="365_day")
feb29s = [str(d) for d in dates if d.month == 2 and d.day == 29]

print(f"Total timesteps: {len(offsets)}")
print(f"Feb 29 count: {len(feb29s)}")

Total timesteps: 27740
Feb 29 count: 0
